# Taller 1 — Programación Dinámica
## Milan Taxi (Versión en Español)

**Curso:** Aprendizaje por Refuerzo  |  **Fecha:** Septiembre 2026

**Objetivo:** Implementar y analizar PI y VI en el entorno MilanTaxi.

## 1. Objetivo

Implementar dos algoritmos de PD (Sutton & Barto Cap. 4):

- **Iteración de Políticas:** Alterna evaluación y mejora.
- **Iteración de Valores:** Computa $V^*$ via el operador de Bellman de optimalidad.

Aplicamos a **MilanTaxi** (cuadrícula 5x5, 500 estados) en dos variantes:
- **Original:** Transiciones deterministas.
- **Estocástica:** Deslizamiento 0.1 en acciones de movimiento.

## 2. Formulación del MDP

Un MDP es $(S, A, P, R, gamma)$.

**Espacio de Estados:** $s$ = (fila, col, pasajero, destino)

| Componente | Rango | Descripción |
|---|---|---|
| Fila | [0,4] | Fila del taxi |
| Columna | [0,4] | Columna del taxi |
| Pasajero | [0,4] | 0=no recogido, 1-4=ubicación |
| Destino | [0,3] | Destino del pasajero |

**Codificación:** fila*100 + col*20 + pasajero*4 + destino

**Acciones:** SUR, NORTE, ESTE, OESTE, RECOGER, DEJAR |A|=6

**Recompensas:** -1 por paso, -10 invalido, +20 DEJAR exitoso

**gamma=0.99**, **Horizonte:** 100 pasos (siempre trunca).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time, sys, os
sys.path.insert(0, os.path.abspath("../src"))
from rl_project.envs.milan_taxi import MilanTaxiEnv,N_S,N_A,HORIZON,encode_state,decode_state,SOUTH,NORTH,EAST,WEST,PICKUP,DROPOFF,LOCS
from rl_project.models.mdp import build_model,check_probability_distribution
from rl_project.agents.dynamic_programming import policy_iteration,value_iteration
from rl_project.evaluation import evaluate_policy
from rl_project import policies
sns.set_theme(style="whitegrid")
%matplotlib inline

GAMMA=0.99
TOL=1e-10
S0=encode_state(0,0,0,1)

## 3. Entorno — Variante Original

Transiciones deterministas.

In [ ]:
env_orig=MilanTaxiEnv(variant="original")
print("Variante:",env_orig.variant,"|S|=",N_S,"|A|=",N_A,"Horizonte=",HORIZON)
state,_=env_orig.reset(seed=42)
print("Estado inicial:",state,"->",decode_state(encode_state(*state)))
print(env_orig.render())
for name,a in [("SUR",0),("ESTE",2),("ESTE",2)]:
    ns,r,_,_,_=env_orig.step(a)
    print(f"{name}: sig={ns}, r={r}")

## 4. Entorno — Variante Estocastica

Deslizamiento: P(intencionado)=0.8, P(izq)=0.1, P(der)=0.1. RECOGER/DEJAR deterministas.

In [ ]:
env_stoch=MilanTaxiEnv(variant="stochastic",slip_probability=0.1)
print("Slip probability:",env_stoch.slip_probability)
print("\n=== Mov. estocastico (ESTE x10) ===")
for t in range(10):
    env_stoch.reset(seed=0)
    ns,r,_,_,_=env_stoch.step(EAST)
    print(f"  Intento {t+1}: sig={ns} -> {decode_state(encode_state(*ns))}")

## 5. Modelo MDP — Construccion de $P$ y $R$

`build_model()` construye $P(s,a,s')$ y $R(s,a)$.

In [ ]:
P_orig,R_orig=build_model(MilanTaxiEnv(variant="original"))
P_stoch,R_stoch=build_model(MilanTaxiEnv(variant="stochastic",slip_probability=0.1))
print("P_orig:",P_orig.shape,"| R_orig:",R_orig.shape)
print("P_stoch:",P_stoch.shape,"| R_stoch:",R_stoch.shape)
assert P_orig.shape==(N_S,N_A,N_S) and R_orig.shape==(N_S,N_A)
assert check_probability_distribution(P_orig) and check_probability_distribution(P_stoch)
assert (P_orig>=0).all() and (P_stoch>=0).all()
assert (P_orig.sum(axis=-1)-1.0<1e-10).all()
print("Validacion P/R: OK")

### 5.1 Original vs Estocasticas

Original: 1 sucesor. Estocastico: hasta 3 sucesores para movimiento.

In [ ]:
s_test=encode_state(2,2,1,3)
print(f"s={decode_state(s_test)}")
a_test=EAST
print(f"a={a_test} (ESTE)")
nz_orig=np.count_nonzero(P_orig[s_test,a_test])
nz_stoch=np.count_nonzero(P_stoch[s_test,a_test])
print(f"  Original: {nz_orig} sucesor(es)")
print(f"  Estoc.: {nz_stoch} sucesor(es)")
print(f"  P_orig={P_orig[s_test,a_test]}")
print(f"  P_stoch={P_stoch[s_test,a_test]}")

## 6. Iteracion de Politicas (PI)

**Algoritmo:** 1) pi0 aleatoria. 2) Evaluar: (I-gamma P^pi) V^pi = R^pi. 3) Mejorar: greedy. 4) Repetir.

In [ ]:
print("=== PI — Original ===")
t0=time.perf_counter()
V_pi_orig,pi_pi_orig,n_pi_orig=policy_iteration(P_orig,R_orig,GAMMA)
t_pi_orig=time.perf_counter()-t0
print(f"  Iter: {n_pi_orig} | V*(0)={V_pi_orig[S0]:.4f} | T={t_pi_orig:.3f}s")
print("\n=== PI — Estocastico ===")
t0=time.perf_counter()
V_pi_stoch,pi_pi_stoch,n_pi_stoch=policy_iteration(P_stoch,R_stoch,GAMMA)
t_pi_stoch=time.perf_counter()-t0
print(f"  Iter: {n_pi_stoch} | V*(0)={V_pi_stoch[S0]:.4f} | T={t_pi_stoch:.3f}s")

### Observaciones PI

- Converge en 7-11 iteraciones.
- Cada iteracion resuelve un sistema 500x500.
- Determinista requiere mas iteraciones (11 vs 7).

## 7. Iteracion de Valores (VI)

Algoritmo:

V_{k+1}(s) = max_a [ R(s,a) + gamma * sum_{s"} P(s"|s,a) V_k(s") ]

Luego extraer politica greedy.

In [ ]:
print("=== VI — Original ===")
t0=time.perf_counter()
V_vi_orig,pi_vi_orig,n_sweeps_orig,deltas_orig=value_iteration(P_orig,R_orig,GAMMA,tol=TOL,max_iter=100000)
t_vi_orig=time.perf_counter()-t0
print(f"  Barridos: {n_sweeps_orig} | V*(0)={V_vi_orig[S0]:.4f} | T={t_vi_orig:.3f}s | df={deltas_orig[-1]:.2e}")
print("\n=== VI — Estocastico ===")
t0=time.perf_counter()
V_vi_stoch,pi_vi_stoch,n_sweeps_stoch,deltas_stoch=value_iteration(P_stoch,R_stoch,GAMMA,tol=TOL,max_iter=100000)
t_vi_stoch=time.perf_counter()-t0
print(f"  Barridos: {n_sweeps_stoch} | V*(0)={V_vi_stoch[S0]:.4f} | T={t_vi_stoch:.3f}s | df={deltas_stoch[-1]:.2e}")

### Observaciones VI

- ~2600 barridos con gamma=0.99, tol=1e-10.
- Cada barrido: O(|S|^2 * |A|).
- Tiempo total ~6s comparable a PI.

## 8. Convergencia de VI

Delta maximo max_s |V_{k+1} - V_k| con los barridos.

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
ax.semilogy(deltas_orig,label="Original",linewidth=2)
ax.semilogy(deltas_stoch,label="Estocastico",linewidth=2,linestyle="--")
ax.axhline(TOL,color="gray",linestyle=":",label=f"tol={TOL}")
ax.set_xlabel("Barrido")
ax.set_ylabel("Delta maximo")
ax.set_title("Convergencia de Value Iteration")
ax.legend()
ax.grid(True,which="both",alpha=0.3)
plt.tight_layout()
plt.savefig("runs/vi_convergence.png",dpi=150,bbox_inches="tight")
plt.show()
print("Guardada en runs/vi_convergence.png")

## 9. Comparacion PI vs VI

Verificar que producen la misma V* y politicas similares.

In [ ]:
print("=== Max diferencia V* ===")
max_diff_orig=np.max(np.abs(V_pi_orig-V_vi_orig))
max_diff_stoch=np.max(np.abs(V_pi_stoch-V_vi_stoch))
print(f"  Original: {max_diff_orig:.2e}")
print(f"  Estoc.: {max_diff_stoch:.2e}")
print("\n=== Acuerdo de politicas ===")
agr_orig=policies.policy_agreement(pi_pi_orig,pi_vi_orig)
agr_stoch=policies.policy_agreement(pi_pi_stoch,pi_vi_stoch)
print(f"  Original: {agr_orig*100:.1f}%")
print(f"  Estoc.: {agr_stoch*100:.1f}%")

## 10. Evaluacion Empirica

1000 episodios Monte Carlo (seed=42, horizonte=100).

In [ ]:
print("Evaluando (1000 eps)...")
eval_res=[]
for n,e,p in [("Orig+PI",MilanTaxiEnv(variant="original"),pi_pi_orig),("Orig+VI",MilanTaxiEnv(variant="original"),pi_vi_orig),("Stoch+PI",MilanTaxiEnv(variant="stochastic",slip=0.1),pi_pi_stoch),("Stoch+VI",MilanTaxiEnv(variant="stochastic",slip=0.1),pi_vi_stoch)]:
    r=evaluate_policy(e,p,n_runs=1000,gamma=GAMMA,seed=42,verbose=False)
    r["name"]=n
    eval_res.append(r)
print("OK")

In [ ]:
print("\n|Variante|gamma|Barridos|V*(0)|Tiempo|")
print("|-|-|-|-|-|")
for l,g,ns,v0,rt in gd:
    print(f"|{l}|{g:.2f}|{ns}|{v0:+.2f}|{rt:.3f}|")

### Obs. sobre gamma

- gamma=0.5: V* negativo (miope).
- gamma=0.9: V* positivo. 248 barridos.
- gamma=0.99: Alto V* pero ~2600 barridos.

Variante estocastica tiene V* mas bajo.

## 12. Preguntas Guia

### Q1: Que es el MDP?
(S,A,P,R,gamma) con 500 estados, 6 acciones, gamma=0.99.

### Q2: Que cambia con transiciones no-deterministas?
P(s'|s,a) pasa de delta a distribucion; V* disminuye.

### Q3: Diferencia fundamental PI vs VI?
PI = dos bucles (Newton), VI = un bucle (descenso).

### Q4: Como sabemos que convergio?
Delta < tol (VI), politica estable (PI), validacion cruzada.

### Q5: Efecto de modificacion estocastica?
V* cae 2.6%, retorno empirico cae ~80%, politica conservadora.

## 13. Limitaciones

1. Horizonte finito (100 pasos) -> brecha teorico-empirica.
2. No terminal.
3. Solo movimiento estocastico.
4. VI ~2600 barridos.
5. Empates en Q (argmax arbitrario).
6. PD tabular no escala.

## 14. Conclusiones

1. MDP critico: 500 estados, 6 acciones, rico y manejable.
2. Estocasticidad reduce retorno (~80% empirico).
3. PI y VI convergen al mismo V*.
4. Trade-off: PI pocas iter caras vs VI muchas baratas.
5. gamma bajo -> miope, gamma alto -> lento.
6. PD tabular no escala.

In [ ]:
print("="*60)
print("EJECUCION DEL NOTEBOOK COMPLETADA")
print("="*60)
print(f"PI it: Orig={n_pi_orig}, Stoch={n_pi_stoch}")
print(f"VI sw: Orig={n_sweeps_orig}, Stoch={n_sweeps_stoch}")
print(f"V*(0) PI orig={V_pi_orig[S0]:.4f}")
print(f"V*(0) VI orig={V_vi_orig[S0]:.4f}")
print(f"V*(0) PI stoch={V_pi_stoch[S0]:.4f}")
print(f"V*(0) VI stoch={V_vi_stoch[S0]:.4f}")
print(f"Acuerdo PI-VI orig={agr_orig*100:.1f}%")
print(f"Acuerdo PI-VI stoch={agr_stoch*100:.1f}%")
print("="*60)